# 04 XAI Model Explanations

This notebook studies the explanation layer for the final alarm set. The goal is not to show a few attractive SHAP plots, but to check whether alarms have understandable evidence across the full five-fold experiment. It loads the all-alarm XAI tables produced by `nita_xai.xai_alarm_episodes`, where every alarm episode representative row is explained with the methods available for its model.

The main questions are: is the alarm evidence recent or old, is it concentrated or diffuse, does it come mostly from speed history or occupancy history, and do SHAP/LIME-style methods or different model families agree? These outputs support the project's central framing: XAI explains model behavior and helps operators judge alarm evidence quality; it does not prove the physical cause of traffic conditions.


## Load All-Alarm XAI Tables

The tables below are based on all alarm episode representative rows across the five folds. A representative row is normally the peak-score timestamp inside an alarm episode, so the explanation corresponds to the moment that best summarizes that episode.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif not (PROJECT_ROOT / "nita_xai").exists():
    PROJECT_ROOT = next(p for p in PROJECT_ROOT.parents if (p / "nita_xai").exists())

sys.path.insert(0, str(PROJECT_ROOT))
from nita_xai import config

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

tables_dir = config.TABLES_DIR
local = pd.read_csv(tables_dir / "xai_local_explanations_all_alarm_episodes.csv")
targets = pd.read_csv(tables_dir / "xai_alarm_episode_targets.csv")
coverage = pd.read_csv(tables_dir / "xai_alarm_episode_coverage.csv")
evidence = pd.read_csv(tables_dir / "xai_alarm_evidence_metrics.csv")
method_agreement = pd.read_csv(tables_dir / "xai_method_agreement_all_alarm_episodes.csv")
cross_model = pd.read_csv(tables_dir / "xai_cross_model_agreement_all_alarm_episodes.csv")

print(f"Alarm episode representative targets: {len(targets):,}")
print(f"Local XAI attribution rows: {len(local):,}")
print(f"Explained case/method combinations: {len(evidence):,}")
print(f"Method-agreement rows: {len(method_agreement):,}")
print(f"Cross-model agreement rows: {len(cross_model):,}")


**Interpretation:** this confirms that the notebook is no longer using only the fold-1 pilot explanations. The evidence table contains one row per explained case and method, while the local table contains the individual lagged feature attributions behind those summaries.


## XAI Coverage

XAI is unavailable for a case when its `representative_row_id`, `model`, `fold`, and `method` are not present in `xai_local_explanations_all_alarm_episodes.csv`. This happens because XAI rows are only present after explicit computation. The coverage table checks this directly.


In [ ]:
coverage_summary = coverage.groupby(["model", "method"]).agg(
    target_cases=("target_cases", "sum"),
    explained_cases=("explained_cases", "sum"),
    missing_cases=("missing_cases", "sum"),
    mean_coverage_rate=("coverage_rate", "mean"),
).reset_index()
coverage_summary


**Interpretation:** complete coverage means the explanation conclusions can be described as patterns across the alarm episode set, not as hand-picked examples. LSTM has sequence attribution only; the project does not add LIME for LSTM because the existing pipeline supports a sequence-suitable DeepSHAP-style method instead.


## What Evidence Do Alarms Use?

This summary aggregates the numerical XAI values into operator-friendly evidence metrics: speed share, occupancy share, recency of evidence, and concentration of the top features. The table is still model-behavior evidence, not a causal traffic diagnosis.


In [ ]:
evidence_summary = evidence.groupby(["model", "method"]).agg(
    cases=("case_id", "nunique"),
    speed_share_mean=("speed_attribution_share", "mean"),
    occupancy_share_mean=("occupancy_attribution_share", "mean"),
    recent_0_15_min_mean=("recent_0_15_min_share", "mean"),
    mid_15_30_min_mean=("mid_15_30_min_share", "mean"),
    older_30_60_min_mean=("older_30_60_min_share", "mean"),
    top3_concentration_mean=("top3_attribution_concentration", "mean"),
    diffuseness_mean=("explanation_diffuseness", "mean"),
).reset_index()
evidence_summary


**Interpretation:** high recent-share values mean the alarm was mainly explained by traffic behavior in the last 0-15 minutes before the representative alarm timestamp. High top-3 concentration means the explanation is focused on a small number of lagged inputs; lower concentration means the evidence is more spread out.


## Evidence Pattern Figures

These plots summarize the full alarm episode set. They translate raw feature attributions into traffic concepts: speed versus occupancy, recent versus older context, and concentrated versus diffuse evidence.


In [ ]:
plot_frame = evidence_summary.set_index(["model", "method"])[[
    "speed_share_mean", "occupancy_share_mean"
]]
ax = plot_frame.plot(kind="bar", stacked=True, figsize=(11, 4))
ax.set_title("Mean attribution share by traffic signal, all alarm episodes")
ax.set_xlabel("Model and explanation method")
ax.set_ylabel("Attribution share")
plt.tight_layout()
plt.show()

recency_frame = evidence_summary.set_index(["model", "method"])[[
    "recent_0_15_min_mean", "mid_15_30_min_mean", "older_30_60_min_mean"
]]
ax = recency_frame.plot(kind="bar", stacked=True, figsize=(11, 4))
ax.set_title("Mean attribution share by temporal recency, all alarm episodes")
ax.set_xlabel("Model and explanation method")
ax.set_ylabel("Attribution share")
plt.tight_layout()
plt.show()


**Interpretation:** these figures answer whether alarms are mostly based on immediate local traffic context or older history. This is more useful than only saying that speed or occupancy mattered.


## Top Lagged Features

The next table shows which lagged inputs most often appear in the top explanations. A feature such as `speed_smoothed_t_minus_1` means speed five minutes before the representative alarm timestamp; `t_minus_3` means fifteen minutes before it.


In [ ]:
top_feature_frequency = (
    local[local["rank"] <= 3]
    .groupby(["model", "method", "feature"])
    .size()
    .reset_index(name="top3_count")
    .sort_values(["model", "method", "top3_count"], ascending=[True, True, False])
)
top_feature_frequency.groupby(["model", "method"]).head(8)


**Interpretation:** repeated recent lag features indicate that the models usually explain alarms through near-term traffic changes. If older lags dominate for a model, that suggests it is using a longer traffic-memory pattern.


## SHAP and LIME Agreement Within Models

For RF, XGBoost, and MLP, this compares the two available explanation styles on the same alarm case. Agreement is defined using top-feature overlap and, where signed values are available, direction agreement. The purpose is not to declare one method correct, but to judge whether the explanation evidence is stable.


In [ ]:
method_counts = method_agreement.groupby([
    "model", "explanation_agreement_level"
]).size().unstack(fill_value=0)
method_counts


In [ ]:
ax = method_counts.plot(kind="bar", stacked=True, figsize=(8, 4))
ax.set_title("SHAP/LIME-style agreement within models, all alarm episodes")
ax.set_xlabel("Model")
ax.set_ylabel("Alarm representative cases")
plt.tight_layout()
plt.show()


**Interpretation:** consistent explanations are easier to use as operator support because different explanation methods point to similar evidence. Partially consistent or conflicting cases are still useful because they warn that the alarm rationale is less stable.


## Cross-Model Explanation Agreement

This table compares explanations when multiple model families alarm on the same timestamp. It checks whether different model families identify similar traffic evidence or whether each model reaches the alarm through different lagged features.


In [ ]:
cross_summary = cross_model.groupby([
    "left_model", "right_model", "explanation_agreement_level"
]).size().reset_index(name="case_count")
cross_summary.pivot_table(
    index=["left_model", "right_model"],
    columns="explanation_agreement_level",
    values="case_count",
    fill_value=0,
)


**Interpretation:** cross-model agreement is important because this project compares model families for XAI, not just for accuracy. Agreement suggests a stable traffic-evidence pattern across models; disagreement suggests a model-specific alarm rationale.


## Final XAI Interpretation

XAI is used as an alarm evidence-quality layer. Score margin and model agreement show how strongly the system alarmed, while SHAP/LIME/DeepSHAP show whether the alarm evidence was recent, concentrated, and consistent. Because XAI was computed for all alarm episode representative rows across the five folds, the resulting patterns describe the alarm set rather than isolated examples.
